# Prophet + semantic-neighbour exogenous features

Upstream source: `sem_prophet.ipynb` in the thesis workbench.

**Input**: `KEYWORDS_DIR_5 / *.parquet` (per-keyword panels with top-5 semantic neighbours).
**Output**: `prophet_results_dir('prophet_semantic_exogdiff')` — `all_metrics.csv` / `.parquet` plus per-cfg per-horizon forecasts under `{univariate,exog_small,exog_full}/h{1,6,12}/forecasts/`.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path("..", "_shared").resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import ensure_env, DATA_ROOT, KEYWORDS_DIR_5, prophet_results_dir  # noqa: E402

ensure_env()


In [ ]:
import pandas as pd
df = pd.read_parquet(DATA_ROOT / "keywords_dfs_full_7" / "yes_rent.parquet")
print(df.columns)
df.head()

In [ ]:
# === TRAIN: Prophet (5 neighbors × {univariate, exog_small, exog_full} × {1,6,12}) ===
# INPUT : KEYWORDS_DIR_5
# OUTPUT: prophet_results_dir("prophet_semantic_exogdiff")
# One consolidated metrics file: all_metrics.csv (+ .parquet)

!pip -q install polars pyarrow tqdm prophet cmdstanpy numpy pandas

from pathlib import Path
from datetime import date
import re
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
from prophet import Prophet

# ---------------- Paths ----------------
BASE   = DATA_ROOT
IN_DIR = KEYWORDS_DIR_5
OUT    = prophet_results_dir("prophet_semantic_exogdiff")
OUT.mkdir(parents=True, exist_ok=True)

# Consolidated metrics output
ALL_METRICS_CSV     = OUT / "all_metrics.csv"
ALL_METRICS_PARQUET = OUT / "all_metrics.parquet"

# Keep forecast CSVs (optional; delete block if you don't need them)
CFG_NAMES = ["univariate", "exog_small", "exog_full"]
HORIZONS  = [1, 6, 12]
FC_DIRS = {(cfg, h): (OUT / cfg / f"h{h}" / "forecasts") for cfg in CFG_NAMES for h in HORIZONS}
for d in FC_DIRS.values(): d.mkdir(parents=True, exist_ok=True)

# ---------------- Config ----------------
K_NEIGH = 5
NEIGHBOR_PREFIX = "cpc_lastweek_"  # single underscore
FIXED_SMALL = [
    "impressions_sum",
    "n_dev_desktop", "n_dev_mobile", "n_dev_tablet",
    "n_st_branded_search", "n_st_generic_search",
]
# exog_full = exog_small + these four extras
FULL_EXTRAS = [
    "avg_sim_top25_this_week", "avg_sim_top25_last_week",
    "n_sim_this_week", "n_sim_last_week",
]

EXCLUDE_ALWAYS = {"week","week_prev","year","week_num","keyword","cpc_week","y","y_log","ds"}
DROP_PREFIXES  = ("adcost", "adclick", "adclocks")  # drop ad spend/clicks
NUM_DTYPES     = (pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64)

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)  # ISO Monday

def smape(y_true, y_pred):
    """
    Symmetric Mean Absolute Percentage Error (SMAPE)
    Defined as:
        SMAPE = 100% * (1/n) * Σ( |y_pred - y_true| / ((|y_true| + |y_pred|) / 2) )
    Handles zero denominators safely.
    """
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    smape_val = np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask])
    return 100 * smape_val

def rmse(y_true, y_pred):
    y_true, y_pred = np.array(y_true, float), np.array(y_pred, float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def make_prophet(add_regressors=None):
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode="additive",
        changepoint_prior_scale=0.05,
        interval_width=0.8,
        stan_backend="CMDSTANPY",
    )
    if add_regressors:
        for r in add_regressors:
            # mild prior shrinkage; Prophet will standardize by default
            m.add_regressor(r, standardize=True, prior_scale=0.2)
    return m

def detect_neighbor_cols(cols, prefix=NEIGHBOR_PREFIX, k=K_NEIGH):
    cols = [c for c in cols if isinstance(c, str)]
    cands = [c for c in cols if c.startswith(prefix)]
    # stable order (numeric suffix if present, else name)
    def sort_key(x):
        m = re.search(r"(\d+)", x)
        return (0 if m else 1, int(m.group(1)) if m else 0, x)
    cands = sorted(cands, key=sort_key)
    return cands[:k]

def coerce_numeric(df_pl: pl.DataFrame, cols):
    present = [c for c in cols if c in df_pl.columns]
    if present:
        df_pl = df_pl.with_columns([pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in present])
    return df_pl

def apply_safe_names(pdf, reg_cols):
    # Prophet handles spaces, but safer to sanitize and avoid dupes
    def safe_name(col: str) -> str:
        s = re.sub(r"\s+", "_", col.strip())
        s = re.sub(r"[^0-9A-Za-z_]", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        return s or "reg"
    name_map = {}
    seen = set()
    for c in reg_cols:
        base = safe_name(c)
        new = base
        k = 2
        while new in seen:
            new = f"{base}_{k}"
            k += 1
        seen.add(new)
        name_map[c] = new
    pdf = pdf.rename(columns=name_map)
    return pdf, [name_map[c] for c in reg_cols]

def prep_frames(pl_df: pl.DataFrame, cfg: str):
    # base ds,y
    base = (
        pl_df.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
             .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
             .drop("week").sort("ds")
             .filter(pl.col("y").is_not_null())
    )
    if cfg == "univariate":
        return base.to_pandas()[["ds","y"]], []

    cols  = pl_df.columns
    neigh = detect_neighbor_cols(cols, NEIGHBOR_PREFIX, K_NEIGH)
    small_present = [c for c in FIXED_SMALL if c in cols]

    if cfg == "exog_small":
        reg_cols = neigh + small_present  # expected ~5 + 6 = 11 (minus any constant per keyword)
    elif cfg == "exog_full":
        extras_present = [c for c in FULL_EXTRAS if c in cols]
        reg_cols = neigh + small_present + extras_present  # expected ~5 + 6 + 4 = 15
    else:
        raise ValueError(f"Unknown cfg: {cfg}")

    # drop any ad spend/clicks that might sneak in (defensive)
    reg_cols = [c for c in reg_cols if not c.lower().startswith(DROP_PREFIXES)]
    if not reg_cols:
        return base.to_pandas()[["ds","y"]], []

    # join exogs on ds
    exdf = (
        pl_df.select(["week"] + reg_cols)
             .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
             .drop("week").sort("ds")
    )
    full = base.join(exdf, on="ds", how="left").to_pandas()

    # numeric coercion + fill
    for c in reg_cols:
        full[c] = pd.to_numeric(full[c], errors="coerce")
    full[reg_cols] = full[reg_cols].replace([np.inf, -np.inf], np.nan).ffill().bfill()

    # drop truly constant regressors (zero variance) per keyword
    nun = full[reg_cols].nunique(dropna=False)
    reg_cols = [c for c in reg_cols if nun.get(c, 0) > 1]
    if not reg_cols:
        return full[["ds","y"]], []

    # sanitize names to be safe
    full, reg_cols = apply_safe_names(full, reg_cols)
    return full[["ds","y"] + reg_cols], reg_cols

def train_eval_save(pdf, reg_cols, kw, cfg, h):
    if len(pdf) <= h + 5:
        return np.nan, np.nan, 0
    train = pdf.iloc[:-h].copy()
    test  = pdf.iloc[-h:].copy()

    m = make_prophet(add_regressors=reg_cols)
    m.fit(train)

    X = test[["ds"] + reg_cols] if reg_cols else test[["ds"]]
    fc = m.predict(X)

    # (optional) write forecasts per cfg/horizon; comment out if not needed
    out_fc = FC_DIRS[(cfg, h)] / f"{kw}.csv"
    pd.DataFrame({
        "ds": test["ds"],
        "y_true": test["y"],
        "yhat": fc["yhat"],
        "yhat_lower": fc.get("yhat_lower", np.nan),
        "yhat_upper": fc.get("yhat_upper", np.nan),
    }).to_csv(out_fc, index=False)

    return smape(test["y"], fc["yhat"]), rmse(test["y"], fc["yhat"]), len(reg_cols)

# ---------------- Train all keywords; write ONE metrics file ----------------
files = sorted(IN_DIR.glob("*.parquet"))
assert files, f"No parquet files found in {IN_DIR}"

records = []  # unified metrics table

for p in tqdm(files, desc="Prophet 5N x (univariate/exog_small/exog_full) x (1/6/12w)"):
    kw = p.stem
    df_pl = pl.read_parquet(p)

    # ensure neighbor columns are numeric before prep (defensive)
    neigh_cols = detect_neighbor_cols(df_pl.columns, NEIGHBOR_PREFIX, K_NEIGH)
    df_pl = coerce_numeric(df_pl, neigh_cols)

    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
        continue

    for cfg in CFG_NAMES:
        pdf, regs = prep_frames(df_pl, cfg)
        for h in HORIZONS:
            s, r, k = train_eval_save(pdf, regs, kw, cfg, h)
            records.append({
                "keyword": kw,
                "horizon_weeks": h,
                "cfg": cfg,
                "smape": s,
                "rmse": r,
                "n_regressors": k,
            })

# Save consolidated metrics
all_metrics = pd.DataFrame(records)
all_metrics["horizon_weeks"] = all_metrics["horizon_weeks"].astype(int)
all_metrics = all_metrics.sort_values(["keyword","horizon_weeks","cfg"])

all_metrics.to_csv(ALL_METRICS_CSV, index=False)
all_metrics.to_parquet(ALL_METRICS_PARQUET, index=False)

# Quick sanity prints
print("\n✅ Training complete.")
print(f"- all metrics (csv):     {ALL_METRICS_CSV}")
print(f"- all metrics (parquet): {ALL_METRICS_PARQUET}")

for cfg in CFG_NAMES:
    dfm = all_metrics[all_metrics["cfg"]==cfg]
    avg_regs = float(dfm["n_regressors"].mean()) if not dfm.empty else 0.0
    target = "0" if cfg=="univariate" else ("15" if cfg=="exog_full" else "11")
    print(f"{cfg}: avg_n_regressors = {avg_regs:.2f} -> expected ~{target}")


In [ ]:
import pandas as pd
from pathlib import Path

OUT = prophet_results_dir("prophet_semantic_exogdiff")
ALL_METRICS_CSV = OUT / "all_metrics.csv"

all_metrics = pd.read_csv(ALL_METRICS_CSV)

# keep only valid rows (in case some keywords were skipped / NaNs)
df = all_metrics.dropna(subset=["smape", "rmse"]).copy()

# nice ordering
cfg_order = ["univariate", "exog_small", "exog_full"]
h_order = [1, 6, 12]
df["cfg"] = pd.Categorical(df["cfg"], categories=cfg_order, ordered=True)
df["horizon_weeks"] = pd.Categorical(df["horizon_weeks"], categories=h_order, ordered=True)

summary = (
    df.groupby(["cfg", "horizon_weeks"], observed=True)
      .agg(
          n_keywords=("keyword", "nunique"),
          smape_mean=("smape", "mean"),
          smape_median=("smape", "median"),
          rmse_mean=("rmse", "mean"),
          rmse_median=("rmse", "median"),
          avg_n_regressors=("n_regressors", "mean"),
      )
      .reset_index()
)

# rounding for readability
for c in ["smape_mean","smape_median","rmse_mean","rmse_median","avg_n_regressors"]:
    summary[c] = summary[c].astype(float).round(3)

summary = summary.sort_values(["cfg","horizon_weeks"])
summary
